In [ ]:
%pip install -qqU diffusers transformers bitsandbytes accelerate ftfy datasets

In [ ]:
import ipywidgets as widgets

theme = "food"
drop_down = widgets.Dropdown(
    options=["animal", "science", "food", "landscape", "wildcard"],
    description="Pick a theme",
    disabled=False,
)


def dropdown_handler(change):
    global theme
    theme = change.new


drop_down.observe(dropdown_handler, names="value")
display(drop_down)

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from pathlib import Path

# Determine whether to use GPU or CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Pick a Stable Diffusion model
model_id = "runwayml/stable-diffusion-v1-5"

In [ ]:
# Load the Stable Diffusion pipeline
# If this is the first time you run it, the model weights will be downloaded.
# For gated models on Hugging Face, make sure you are logged in or have set HUGGINGFACE_HUB_TOKEN.

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)

if device == "cuda":
    pipe = pipe.to("cuda")

print("Pipeline ready.")

In [ ]:
import random

# Prompts for each theme
theme_prompts = {
    "food": [
        "highly detailed photo of gourmet pasta on a rustic wooden table, soft cinematic lighting",
        "macro shot of colorful sushi assortment on a slate plate, studio lighting, 4k, ultra realistic",
        "stack of fluffy pancakes with maple syrup and berries, morning light, food photography",
    ],
    "animal": [
        "portrait of a red fox in a forest, shallow depth of field, 85mm lens, bokeh",
        "majestic lion at golden hour on the savannah, dramatic lighting, ultra realistic",
        "cute baby panda sitting in bamboo forest, soft lighting, award-winning wildlife photograph",
    ],
    "science": [
        "futuristic laboratory interior, holographic user interfaces, blue and purple neon lighting",
        "cross-section illustration of a human cell, vibrant colors, detailed scientific illustration",
        "space station orbiting Earth, ultra detailed, cinematic, volumetric lighting",
    ],
    "landscape": [
        "epic mountain range at sunrise, clouds in valley, ultra wide angle, 8k",
        "serene lake surrounded by autumn forest, mirror reflection, hyper realistic",
        "coastal cliffs at sunset, crashing waves, long exposure photography, dramatic sky",
    ],
    "wildcard": [
        "surreal dreamscape of floating islands and waterfalls in the sky, fantasy art, highly detailed",
        "cyberpunk city street at night, neon signs, rain-soaked pavement, cinematic lighting",
        "ancient mystical library filled with glowing books, volumetric light rays, ultra detailed",
    ],
}

# How many images to generate per run
num_images = 4

# Generation parameters
guidance_scale = 7.5
num_inference_steps = 30

print(f"Current theme is: {theme}")

In [ ]:
# Generate and save images for the selected theme

from IPython.display import display

# Ensure output directory exists
output_dir = Path("outputs") / theme
output_dir.mkdir(parents=True, exist_ok=True)

selected_prompts = theme_prompts.get(theme, theme_prompts["wildcard"])

results = []
for i in range(num_images):
    prompt = random.choice(selected_prompts)
    seed = random.randint(0, 10_000)
    generator = torch.Generator(device=device).manual_seed(seed)

    image = pipe(
        prompt,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        generator=generator,
    ).images[0]

    file_path = output_dir / f"{theme}_{i+1}_seed{seed}.png"
    image.save(file_path)

    results.append({
        "image": image,
        "prompt": prompt,
        "seed": seed,
        "path": file_path,
    })

print(f"Saved {len(results)} images to {output_dir.resolve()}")

# Display the generated images with prompts and metadata
for r in results:
    display(r["image"])
    print(f"Prompt: {r['prompt']}")
    print(f"Seed:   {r['seed']}")
    print(f"File:   {r['path']}")
    print("-" * 60)